In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Chaotic Associative Memory Cryptography\n",
    "## Block Cipher Security Analysis\n",
    "\n",
    "Statistical evaluation of the 64-byte Feistel block cipher.\n",
    "Uses a chaotic oscillatory neural network as the round function.\n",
    "Metrics: byte distribution, Shannon entropy, avalanche effect,\n",
    "autocorrelation, and integrity verification."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from network import ChaoticOscillatoryNetwork\n",
    "from block_cipher import CAMCBlockCipher\n",
    "from dynamics import entropy, autocorrelation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.rcParams.update({\n",
    "    'figure.facecolor': '#fafafa',\n",
    "    'axes.facecolor': '#fafafa',\n",
    "    'axes.edgecolor': '#222222',\n",
    "    'axes.labelcolor': '#222222',\n",
    "    'text.color': '#222222',\n",
    "    'font.size': 11,\n",
    "    'axes.grid': True,\n",
    "    'grid.alpha': 0.3,\n",
    "    'grid.color': '#999999'\n",
    "})"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "N_NEURONS = 128\n",
    "WEIGHT_SCALE = 1.5\n",
    "ROUNDS = 8\n",
    "STEPS_PER_ROUND = 2\n",
    "\n",
    "np.random.seed(42)\n",
    "network = ChaoticOscillatoryNetwork(n_neurons=N_NEURONS, weight_scale=WEIGHT_SCALE)\n",
    "cipher = CAMCBlockCipher(network, rounds=ROUNDS, steps_per_round=STEPS_PER_ROUND)\n",
    "\n",
    "plain_64 = b\"A\" * 64\n",
    "plain_32 = b\"B\" * 32\n",
    "pattern = b\"CHKSUM1234CHECKSABCDEFGHIJKLMNOP\""
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Byte Histogram (64-byte block, no integrity)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "encrypted_64 = cipher.encrypt(plain_64)\n",
    "cipher_bytes = np.frombuffer(encrypted_64, dtype=np.uint8)\n",
    "plain_bytes = np.frombuffer(plain_64, dtype=np.uint8)\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "axes[0].hist(plain_bytes, bins=256, color='#2d4a2b', alpha=0.9)\n",
    "axes[0].set_title('Plaintext Byte Distribution')\n",
    "axes[0].set_xlabel('Byte value')\n",
    "axes[0].set_ylabel('Count')\n",
    "axes[1].hist(cipher_bytes, bins=256, color='#4a7c59', alpha=0.9)\n",
    "axes[1].set_title('Ciphertext Byte Distribution')\n",
    "axes[1].set_xlabel('Byte value')\n",
    "axes[1].set_ylabel('Count')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Shannon Entropy"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plain_entropy = entropy(plain_64)\n",
    "cipher_entropy = entropy(encrypted_64)\n",
    "print(f'Plaintext entropy : {plain_entropy:.4f} bits/byte')\n",
    "print(f'Ciphertext entropy: {cipher_entropy:.4f} bits/byte')\n",
    "print(f'Max possible      : 8.0000 bits/byte')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Avalanche Effect (64-byte block)\n",
    "Change one bit in plaintext, measure output bit difference."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "p1 = plain_64\n",
    "p2 = bytearray(p1)\n",
    "p2[0] ^= 0x01\n",
    "p2 = bytes(p2)\n",
    "c1 = cipher.encrypt(p1)\n",
    "c2 = cipher.encrypt(p2)\n",
    "diff_bits = sum(bin(a ^ b).count('1') for a, b in zip(c1, c2))\n",
    "total_bits = len(c1) * 8\n",
    "print(f'Changed bits: {diff_bits}/{total_bits} = {100*diff_bits/total_bits:.1f}%')\n",
    "print('Ideal: 50%')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Autocorrelation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ac = autocorrelation(encrypted_64, max_lag=30)\n",
    "plt.figure(figsize=(10, 3))\n",
    "plt.stem(range(len(ac)), ac)\n",
    "plt.title('Ciphertext Autocorrelation')\n",
    "plt.xlabel('Lag (bytes)')\n",
    "plt.ylabel('Correlation coefficient')\n",
    "plt.axhline(0, color='#999999', linewidth=0.5)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Integrity Verification (32-byte message + 32-byte pattern)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "cipher.set_integrity_pattern(pattern)\n",
    "encrypted_32 = cipher.encrypt(plain_32)\n",
    "decrypted_32 = cipher.decrypt(encrypted_32)\n",
    "print(f'Original : {plain_32}')\n",
    "print(f'Decrypted: {decrypted_32}')\n",
    "print(f'Match: {decrypted_32 == plain_32}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "tampered = bytearray(encrypted_32)\n",
    "tampered[10] ^= 0xFF\n",
    "try:\n",
    "    cipher.decrypt(bytes(tampered))\n",
    "    print('ERROR: Tampering not detected')\n",
    "except ValueError as e:\n",
    "    print(f'Tampering correctly detected: {e}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Summary\n",
    "\n",
    "The 64-byte Feistel block cipher with chaotic round function produces\n",
    "ciphertext with high entropy, uniform byte distribution, near-optimal\n",
    "avalanche effect, and negligible autocorrelation. The associative memory\n",
    "based integrity check reliably detects tampering."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}